# ComfyUI Telegram Бот — Colab T4

Запустите **Telegram бота**, который генерирует видео и фото с помощью ComfyUI на бесплатном GPU T4.

```
╔══════════╗    ╔══════════════╗    ╔══════════╗    ╔════════════╗    ╔═══════════════╗
║  1. GPU  ║ -> ║ 2. Установка ║ -> ║ 3.Модели ║ -> ║ 4.Воркфлоу ║ -> ║ 5. ЗАПУСК БОТА║ -> Telegram!
╚══════════╝    ╚══════════════╝    ╚══════════╝    ╚════════════╝    ╚═══════════════╝
    ~5 сек         ~2 мин             ~10 мин          ~30 сек            ~1 мин
```

---

### Что нужно до запуска?

> **Получите токен бота у [@BotFather](https://t.me/BotFather) в Telegram — это займёт 1 минуту:**
>
> 1. Откройте [@BotFather](https://t.me/BotFather) в Telegram
> 2. Отправьте `/newbot`
> 3. Придумайте **имя** бота (например: `Мой Видео Бот`)
> 4. Придумайте **username** (например: `my_comfy_video_bot`)
> 5. Скопируйте **токен** вида `1234567890:ABCdefGhIjKlMnOpQrStUvWxYz`
>
> Токен понадобится в ячейке 5.

---

### Команды бота

| Команда | Что отправить | Воркфлоу | Результат |
|:--------|:--------------|:---------|:----------|
| `/photo <промпт>` | Текст + ответ на фото | `wan_clip` | Короткое видео 3.4с из изображения |
| `/text <промпт>` | Только текст | `wan_t2v` | Видео из текстового описания |
| `/talking` | Ответ на фото + аудио | `wan_talking` | Говорящая голова |
| `/v2v <промпт>` | Текст + ответ на видео | `wan_v2v` | Рестайл видео |
| `/status` | — | — | Проверить статус очереди |
| `/cancel` | — | — | Отменить текущую задачу |

### Ограничения

| Параметр | Значение |
|:---------|:---------|
| Время сессии | Макс. 12ч, отключение через 90мин простоя |
| Генерация | 2-10 мин на видео (T4 16ГБ) |
| Параллельность | Одна задача за раз (ограничение VRAM) |
| Размер видео | Telegram лимит — 50 МБ |

---

**Запускайте ячейки 1-5 по порядку.**

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды + Зависимости бота
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

# Кастомные ноды (те же, что в wan ноутбуке)
%cd /content/ComfyUI/custom_nodes
!test -d ComfyUI-WanVideoWrapper || git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!test -d ComfyUI-VideoHelperSuite || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git

!pip install -r ComfyUI-WanVideoWrapper/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-KJNodes/requirements.txt -q -c /tmp/numpy_constraint.txt

# Зависимости бота
!pip install python-telegram-bot==21.6 requests -q

print("\nВсё установлено!")

In [ ]:
#@title 3. Скачивание моделей (~28 ГБ)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

HF = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    f"{M}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{HF}/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    f"{M}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors":
        f"{HF}/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
    f"{M}/diffusion_models/fantasytalking_fp16.safetensors":
        f"{HF}/fantasytalking_fp16.safetensors",
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{HF}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    f"{M}/vae/wan2.2_vae.safetensors":
        f"{HF}/wan2.2_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
for dest in files:
    if os.path.exists(dest) and os.path.getsize(dest) < 1024:
        os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

print("\nВсе модели готовы!")

In [ ]:
#@title 4. Скачивание воркфлоу (API-формат для бота)
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
REPO_RAW = "https://raw.githubusercontent.com/russiankendricklamar/comfyui-colab-toolkit/main/workflows"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

# UI-воркфлоу (для ручной работы в интерфейсе)
ui_workflows = [
    "video_wan_t2v.json",
    "video_wan_clip.json",
    "video_wan_v2v.json",
    "video_wan_talking.json",
]

# API-воркфлоу (для автоматической генерации ботом)
api_workflows = [
    "video_wan_t2v_api.json",
    "video_wan_clip_api.json",
    "video_wan_v2v_api.json",
    "video_wan_talking_api.json",
]

for wf in ui_workflows:
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}"
    print(f"  UI: {wf}")

for wf in api_workflows:
    # Пробуем gist, затем GitHub repo
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}" 2>/dev/null || wget -q -O "{WF_DIR}/{wf}" "{REPO_RAW}/{wf}"
    if os.path.exists(f"{WF_DIR}/{wf}") and os.path.getsize(f"{WF_DIR}/{wf}") > 100:
        print(f"  API: {wf}")
    else:
        print(f"  ОШИБКА: {wf} не найден — бот будет работать в ручном режиме")

print(f"\n{len(ui_workflows) + len(api_workflows)} воркфлоу готовы")

In [ ]:
#@title 5. Настройка и запуск бота
#@markdown ---
#@markdown ### Способ подключения к ComfyUI
#@markdown **ngrok** — самый стабильный.
tunnel_method = "ngrok" #@param ["ngrok", "cloudflare", "localtunnel"]
ngrok_token = "" #@param {type:"string"}
#@markdown > Токен ngrok: [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
#@markdown
#@markdown ---
#@markdown ### Как получить токен бота (1 минута)
#@markdown
#@markdown 1. Откройте [@BotFather](https://t.me/BotFather) в Telegram
#@markdown 2. Отправьте `/newbot`
#@markdown 3. Придумайте имя и username для бота
#@markdown 4. Скопируйте токен вида `1234567890:ABCdefGhIjKlMnOpQrStUvWxYz`
#@markdown 5. Вставьте его в поле ниже
#@markdown
#@markdown ---
#@markdown ### Белый список пользователей
#@markdown Укажите ваш Telegram ID, чтобы бот отвечал только вам.
#@markdown Узнать ID: отправьте `/start` боту [@userinfobot](https://t.me/userinfobot)
#@markdown Оставьте пустым, чтобы бот отвечал всем.
#@markdown
#@markdown ---
#@markdown ### Как работает бот
#@markdown ```
#@markdown Telegram        Bot           ComfyUI API     Telegram
#@markdown   |              |               |               |
#@markdown   |-- команда -->|               |               |
#@markdown   |              |-- /api/prompt->|               |
#@markdown   |              |               |-- генерация   |
#@markdown   |              |  poll history  |   (2-10 мин)  |
#@markdown   |              |<-- готово -----|               |
#@markdown   |              |--------------- видео -------->|
#@markdown ```
#@markdown
#@markdown ---
#@markdown ### Что произойдёт после запуска
#@markdown - Запустится **ComfyUI** на порту 8188 (~30 сек)
#@markdown - Откроется **туннель** для веб-интерфейса
#@markdown - Запустится **Telegram бот** в режиме polling
#@markdown - В выводе появится ссылка на интерфейс ComfyUI
#@markdown - Отправьте `/start` боту в Telegram для начала!

import getpass
BOT_TOKEN = getpass.getpass("Вставьте токен Telegram бота: ")
_whitelist_input = input("Telegram ID для белого списка (через запятую, или Enter для доступа всем): ").strip()
ALLOWED_USERS = {int(x.strip()) for x in _whitelist_input.split(",") if x.strip().isdigit()} if _whitelist_input else set()

import subprocess, time, os, json, glob, asyncio, re, shutil, requests, copy
from telegram import Update
from telegram.ext import Application, CommandHandler, ContextTypes

# ── Константы ────────────────────────────────────────────────────────

COMFY_URL = "http://127.0.0.1:8188"
INPUT_DIR = "/content/ComfyUI/input"
OUTPUT_DIR = "/content/ComfyUI/output"
WF_DIR = "/content/ComfyUI/user/default/workflows"

# Маппинг воркфлоу -> API файл и ключевые ноды
# Структура: {workflow: {api_file, prompt_node, prompt_field, image_node, image_field, ...}}
WORKFLOW_MAP = {
    "video_wan_t2v": {
        "api_file": "video_wan_t2v_api.json",
        "prompt_node": "20", "prompt_field": "positive",
    },
    "video_wan_clip": {
        "api_file": "video_wan_clip_api.json",
        "prompt_node": "20", "prompt_field": "positive",
        "image_node": "2", "image_field": "image",
    },
    "video_wan_talking": {
        "api_file": "video_wan_talking_api.json",
        "prompt_node": "20", "prompt_field": "positive",
        "image_node": "2", "image_field": "image",
        "audio_node": "70", "audio_field": "audio",
    },
    "video_wan_v2v": {
        "api_file": "video_wan_v2v_api.json",
        "prompt_node": "6", "prompt_field": "positive",
        "video_node": "1", "video_field": "video",
    },
}

# ── Состояние ────────────────────────────────────────────────────────

job_lock = asyncio.Lock()
current_job = {"active": False, "user_id": None, "prompt_id": None}

# ── ComfyUI API ──────────────────────────────────────────────────────

def load_workflow(workflow_key):
    """Загрузка API-воркфлоу из файла."""
    info = WORKFLOW_MAP.get(workflow_key)
    if not info:
        return None, None
    api_path = os.path.join(WF_DIR, info["api_file"])
    if not os.path.exists(api_path):
        return None, info
    with open(api_path, "r") as f:
        return json.load(f), info

def inject_params(workflow, info, prompt_text=None, image_name=None, audio_name=None, video_name=None):
    """Подстановка параметров пользователя в воркфлоу."""
    wf = copy.deepcopy(workflow)
    if prompt_text and "prompt_node" in info:
        wf[info["prompt_node"]]["inputs"][info["prompt_field"]] = prompt_text
    if image_name and "image_node" in info:
        wf[info["image_node"]]["inputs"][info["image_field"]] = image_name
    if audio_name and "audio_node" in info:
        wf[info["audio_node"]]["inputs"][info["audio_field"]] = audio_name
    if video_name and "video_node" in info:
        wf[info["video_node"]]["inputs"][info["video_field"]] = video_name
    # Рандомизация seed
    for node_id, node in wf.items():
        if "seed" in node.get("inputs", {}):
            import random
            wf[node_id]["inputs"]["seed"] = random.randint(0, 2**32 - 1)
    return wf

def queue_prompt(workflow):
    """Отправка воркфлоу в ComfyUI API."""
    resp = requests.post(
        f"{COMFY_URL}/api/prompt",
        json={"prompt": workflow},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()["prompt_id"]

def get_history(prompt_id):
    """Получение истории выполнения по prompt_id."""
    resp = requests.get(f"{COMFY_URL}/api/history/{prompt_id}", timeout=10)
    if resp.status_code == 200:
        data = resp.json()
        return data.get(prompt_id)
    return None

def find_output_files(history_entry):
    """Извлечение путей к результатам из истории ComfyUI."""
    files = []
    if not history_entry or "outputs" not in history_entry:
        return files
    for node_id, node_output in history_entry["outputs"].items():
        for key in ("gifs", "images", "videos"):
            for item in node_output.get(key, []):
                fname = item.get("filename", "")
                subfolder = item.get("subfolder", "")
                ftype = item.get("type", "output")
                if ftype == "output":
                    fpath = os.path.join(OUTPUT_DIR, subfolder, fname) if subfolder else os.path.join(OUTPUT_DIR, fname)
                    if os.path.exists(fpath):
                        files.append(fpath)
    return files

# ── Проверка доступа ─────────────────────────────────────────────────

def check_access(update: Update) -> bool:
    if not ALLOWED_USERS:
        return True
    return update.effective_user.id in ALLOWED_USERS

# ── Обработчики бота ─────────────────────────────────────────────────

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    await update.message.reply_text(
        "ComfyUI Видео Бот\n\n"
        "Команды:\n"
        "/photo <промпт> — ответьте на фото, чтобы анимировать его (клип 3.4с)\n"
        "/text <промпт> — генерация видео из текста\n"
        "/talking — ответьте на фото, прикрепите аудио\n"
        "/v2v <промпт> — ответьте на видео для рестайла\n"
        "/status — проверить текущую задачу\n"
        "/cancel — отменить текущую задачу\n\n"
        "Одна задача за раз. Генерация занимает 2-10 мин."
    )

async def status_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    async with job_lock:
        if current_job["active"]:
            pid = current_job["prompt_id"]
            pid_short = pid[:8] if pid else "?"
            await update.message.reply_text(f"Задача выполняется (prompt_id: {pid_short}...)")
        else:
            await update.message.reply_text("Нет активных задач. Отправьте команду для начала!")

async def cancel_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    async with job_lock:
        if current_job["active"]:
            try:
                await asyncio.to_thread(requests.post, f"{COMFY_URL}/interrupt", timeout=5)
                current_job["active"] = False
                current_job["prompt_id"] = None
                await update.message.reply_text("Задача отменена.")
            except Exception as e:
                await update.message.reply_text(f"Ошибка отмены: {e}")
        else:
            await update.message.reply_text("Нет активной задачи для отмены.")

async def generate_video(update, context, workflow_key, prompt_text,
                         image_name=None, audio_name=None, video_name=None):
    """Отправка задачи в ComfyUI API и ожидание результата."""
    async with job_lock:
        if current_job["active"]:
            await update.message.reply_text("Другая задача уже выполняется. Используйте /status или /cancel.")
            return
        current_job["active"] = True
        current_job["user_id"] = update.effective_user.id

    try:
        # Загрузка API-воркфлоу
        workflow, info = load_workflow(workflow_key)
        if workflow is None:
            await update.message.reply_text(
                f"API-воркфлоу {WORKFLOW_MAP.get(workflow_key, {}).get('api_file', '?')} не найден.\n"
                "Откройте ComfyUI через туннель и запустите генерацию вручную.\n"
                "Бот будет отслеживать папку вывода..."
            )
            # Фоллбек на мониторинг папки
            await _fallback_monitor(update, workflow_key)
            return

        # Подстановка параметров
        wf = inject_params(workflow, info, prompt_text, image_name, audio_name, video_name)

        status_msg = await update.message.reply_text(f"Отправка в ComfyUI ({workflow_key})...")

        # Отправка в ComfyUI API
        try:
            prompt_id = await asyncio.to_thread(queue_prompt, wf)
        except Exception as e:
            await status_msg.edit_text(f"Ошибка отправки в ComfyUI: {e}")
            return

        async with job_lock:
            current_job["prompt_id"] = prompt_id

        await status_msg.edit_text(f"В очереди! Генерация... (2-10 мин)\nID: {prompt_id[:8]}...")

        # Ожидание завершения через /api/history
        start_time = time.time()
        timeout = 600  # 10 мин
        while time.time() - start_time < timeout:
            await asyncio.sleep(10)
            async with job_lock:
                if not current_job["active"]:
                    return  # Отменено
            try:
                history = await asyncio.to_thread(get_history, prompt_id)
            except Exception:
                continue
            if history and history.get("status", {}).get("completed", False):
                # Генерация завершена
                output_files = find_output_files(history)
                if output_files:
                    for fpath in output_files:
                        try:
                            if fpath.endswith('.mp4'):
                                with open(fpath, 'rb') as f:
                                    await update.message.reply_video(
                                        video=f,
                                        caption=f"Сгенерировано ({workflow_key})"
                                    )
                            elif fpath.endswith(('.png', '.jpg', '.jpeg', '.gif')):
                                with open(fpath, 'rb') as f:
                                    await update.message.reply_photo(
                                        photo=f,
                                        caption=f"Сгенерировано ({workflow_key})"
                                    )
                        except Exception as e:
                            await update.message.reply_text(f"Ошибка отправки файла: {e}")
                    await status_msg.edit_text("Готово!")
                else:
                    await status_msg.edit_text(
                        "Генерация завершена, но файлы не найдены.\n"
                        "Проверьте интерфейс ComfyUI."
                    )
                return
            # Проверка ошибок в статусе
            if history and history.get("status", {}).get("status_str") == "error":
                msgs = history.get("status", {}).get("messages", [])
                err_text = str(msgs[-1]) if msgs else "неизвестная ошибка"
                await status_msg.edit_text(f"Ошибка ComfyUI: {err_text}")
                return

        await status_msg.edit_text("Генерация превысила таймаут (10 мин). Проверьте интерфейс ComfyUI.")

    except Exception as e:
        await update.message.reply_text(f"Ошибка: {e}")
    finally:
        async with job_lock:
            current_job["active"] = False
            current_job["prompt_id"] = None

async def _fallback_monitor(update, workflow_key):
    """Мониторинг папки вывода (фоллбек если API-воркфлоу нет)."""
    existing_files = set(glob.glob(f"{OUTPUT_DIR}/*.*"))
    start_time = time.time()
    timeout = 600
    while time.time() - start_time < timeout:
        await asyncio.sleep(10)
        async with job_lock:
            if not current_job["active"]:
                return
        current_files = set(glob.glob(f"{OUTPUT_DIR}/*.*"))
        new_files = current_files - existing_files
        new_media = [f for f in new_files if f.endswith(('.mp4', '.png', '.gif'))]
        if new_media:
            for fpath in sorted(new_media):
                try:
                    if fpath.endswith('.mp4'):
                        with open(fpath, 'rb') as f:
                            await update.message.reply_video(
                                video=f,
                                caption=f"Сгенерировано ({workflow_key})"
                            )
                    else:
                        with open(fpath, 'rb') as f:
                            await update.message.reply_photo(
                                photo=f,
                                caption=f"Сгенерировано ({workflow_key})"
                            )
                except Exception as e:
                    await update.message.reply_text(f"Ошибка отправки файла: {e}")
            return

async def cmd_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    prompt = " ".join(context.args) if context.args else None
    if not prompt:
        await update.message.reply_text("Использование: /text <ваш промпт>")
        return
    await generate_video(update, context, "video_wan_t2v", prompt)

async def cmd_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    prompt = " ".join(context.args) if context.args else "animate this image, smooth motion, cinematic"
    reply = update.message.reply_to_message
    photo_file = None

    if reply and reply.photo:
        photo_file = await reply.photo[-1].get_file()
    elif update.message.photo:
        photo_file = await update.message.photo[-1].get_file()

    if not photo_file:
        await update.message.reply_text("Ответьте на фото командой /photo <промпт> или отправьте фото вместе с командой.")
        return

    fname = f"bot_input_{int(time.time())}.jpg"
    fpath = os.path.join(INPUT_DIR, fname)
    await photo_file.download_to_drive(fpath)
    await generate_video(update, context, "video_wan_clip", prompt, image_name=fname)

async def cmd_talking(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    reply = update.message.reply_to_message
    photo_file = None
    audio_file = None

    if reply and reply.photo:
        photo_file = await reply.photo[-1].get_file()

    if update.message.audio:
        audio_file = await update.message.audio.get_file()
    elif update.message.voice:
        audio_file = await update.message.voice.get_file()
    elif reply and reply.audio:
        audio_file = await reply.audio.get_file()
    elif reply and reply.voice:
        audio_file = await reply.voice.get_file()

    if not photo_file or not audio_file:
        await update.message.reply_text(
            "Для говорящей головы:\n"
            "1. Отправьте портретное фото\n"
            "2. Ответьте на него командой /talking и прикрепите аудиофайл/голосовое сообщение"
        )
        return

    ts = int(time.time())
    photo_name = f"bot_photo_{ts}.jpg"
    audio_name = f"bot_audio_{ts}.wav"
    await photo_file.download_to_drive(os.path.join(INPUT_DIR, photo_name))
    await audio_file.download_to_drive(os.path.join(INPUT_DIR, audio_name))

    await generate_video(update, context, "video_wan_talking",
                         "talking head, natural expressions",
                         image_name=photo_name, audio_name=audio_name)

async def cmd_v2v(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not check_access(update):
        return
    prompt = " ".join(context.args) if context.args else None
    if not prompt:
        await update.message.reply_text("Использование: ответьте на видео командой /v2v <промпт>")
        return

    reply = update.message.reply_to_message
    video_file = None
    if reply and reply.video:
        video_file = await reply.video.get_file()
    elif reply and reply.document and reply.document.mime_type and reply.document.mime_type.startswith("video"):
        video_file = await reply.document.get_file()

    if not video_file:
        await update.message.reply_text("Ответьте на видео командой /v2v <промпт>")
        return

    ts = int(time.time())
    video_name = f"bot_video_{ts}.mp4"
    await video_file.download_to_drive(os.path.join(INPUT_DIR, video_name))

    await generate_video(update, context, "video_wan_v2v", prompt, video_name=video_name)

# ── Запуск ───────────────────────────────────────────────────────────

if not BOT_TOKEN:
    print("ОШИБКА: Установите BOT_TOKEN выше!")
else:
    # Запуск ComfyUI
    print("=" * 60)
    print("ЭТАП 1/3: Запуск ComfyUI...")
    print("=" * 60)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # Убиваем предыдущий экземпляр ComfyUI если есть
    !kill -9 $(lsof -t -i:8188) 2>/dev/null || true

    comfy_proc = subprocess.Popen(
        ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188",
         "--enable-cors-header", "*"],
        cwd="/content/ComfyUI",
        stdout=open("/content/comfyui.log", "w"),
        stderr=subprocess.STDOUT
    )

    print("  Ожидание запуска ComfyUI...")
    for _i in range(24):
        time.sleep(5)
        try:
            r = requests.get(f"{COMFY_URL}/api/system_stats", timeout=3)
            if r.status_code == 200:
                gpu_name = r.json()['devices'][0]['name']
                print(f"  ComfyUI запущен за {(_i+1)*5} сек! GPU: {gpu_name}")
                break
        except Exception:
            pass
    else:
        print("  ComfyUI не ответил за 120 сек — проверьте /content/comfyui.log")
        with open("/content/comfyui.log") as _f:
            _log = _f.read()
        _errs = [l for l in _log.splitlines() if "error" in l.lower() or "traceback" in l.lower()]
        if _errs:
            print("  Возможные ошибки в логе:")
            for _e in _errs[:10]:
                print(f"    {_e}")

    # === ТУННЕЛЬ ===
    print("\n" + "=" * 60)
    print(f"ЭТАП 2/3: Открытие туннеля ({tunnel_method})...")
    print("=" * 60)
    tunnel_url = None

    if tunnel_method == "ngrok":
        !pip install pyngrok -q
        from pyngrok import ngrok
        if not ngrok_token:
            import getpass as _gp
            ngrok_token = _gp.getpass("Введите ngrok authtoken (dashboard.ngrok.com → Your Authtoken): ")
        ngrok.set_auth_token(ngrok_token)
        _tunnel = ngrok.connect(8188)
        tunnel_url = _tunnel.public_url

    elif tunnel_method == "cloudflare":
        if not shutil.which("cloudflared"):
            subprocess.run(
                ["bash", "-c",
                 "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1"],
                check=False
            )
        tunnel = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        tunnel_url = None
        time.sleep(5)
        for _ in range(20):
            line = tunnel.stdout.readline().decode()
            match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
            if match:
                tunnel_url = match.group(0)
                break

    elif tunnel_method == "localtunnel":
        subprocess.run(["npm", "install", "-g", "localtunnel"], capture_output=True)
        tunnel = subprocess.Popen(
            ["lt", "--port", "8188"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        tunnel_url = None
        time.sleep(8)
        for _ in range(20):
            line = tunnel.stdout.readline().decode()
            match = re.search(r"https://[\w-]+\.loca\.lt", line)
            if match:
                tunnel_url = match.group(0)
                break

    if tunnel_url:
        print(f"  Веб-интерфейс ComfyUI: {tunnel_url}")
        print("  (используйте для отладки и ручного запуска воркфлоу)")
    else:
        print("  Туннель не открылся. Бот всё равно будет работать.")
        print("  Для доступа к UI перезапустите ячейку 5.")

    # Проверка API-воркфлоу
    api_ok = 0
    for key, info in WORKFLOW_MAP.items():
        if os.path.exists(os.path.join(WF_DIR, info["api_file"])):
            api_ok += 1
    if api_ok == len(WORKFLOW_MAP):
        print(f"\n  Все {api_ok} API-воркфлоу загружены — бот работает в автоматическом режиме!")
    else:
        print(f"\n  Загружено {api_ok}/{len(WORKFLOW_MAP)} API-воркфлоу.")
        print("  Недостающие воркфлоу: бот будет отслеживать папку вывода (ручной режим).")

    print("\n" + "=" * 60)
    print("ЭТАП 3/3: Запуск Telegram бота...")
    print("=" * 60)
    if ALLOWED_USERS:
        print(f"  Белый список: {ALLOWED_USERS}")
    else:
        print("  Белый список: ОТКЛЮЧЁН (бот отвечает всем)")
    print()
    print("-" * 60)
    print("  ГОТОВО! Откройте бота в Telegram и отправьте /start")
    print("  Для остановки нажмите кнопку СТОП в Colab")
    print("-" * 60)
    print()

    app = Application.builder().token(BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("text", cmd_text))
    app.add_handler(CommandHandler("photo", cmd_photo))
    app.add_handler(CommandHandler("talking", cmd_talking))
    app.add_handler(CommandHandler("v2v", cmd_v2v))
    app.add_handler(CommandHandler("status", status_cmd))
    app.add_handler(CommandHandler("cancel", cancel_cmd))

    app.run_polling(allowed_updates=Update.ALL_TYPES)

---

## Руководство по использованию

### Подготовка (до запуска ноутбука)

| Шаг | Действие | Время |
|:---:|:---------|:-----:|
| 1 | Откройте [@BotFather](https://t.me/BotFather) в Telegram | 10 сек |
| 2 | Отправьте `/newbot`, придумайте имя и username | 30 сек |
| 3 | Скопируйте токен (понадобится в ячейке 5) | 10 сек |
| 4 | Узнайте свой Telegram ID: [@userinfobot](https://t.me/userinfobot) | 10 сек |

---

### Как это работает

```
  ВЫ (Telegram)                    Google Colab
  ┌───────────┐              ┌──────────────────────┐
  │           │  /photo кот  │   ┌───────────────┐   │
  │ Telegram  │ -----------> │   │  Telegram бот  │   │
  │  клиент   │              │   └──────┬────────┘   │
  │           │              │          |             │
  │           │              │          v             │
  │           │              │   ┌───────────────┐   │
  │           │              │   │ ComfyUI API    │   │
  │           │              │   │ /api/prompt    │   │
  │           │              │   │ (генерация     │   │
  │           │              │   │  2-10 мин)     │   │
  │           │              │   └──────┬────────┘   │
  │           │   .mp4       │          |             │
  │           │ <----------- │   /api/history         │
  └───────────┘              └──────────────────────┘
```

**Пошагово:**
1. Вы отправляете команду боту в Telegram (например, `/photo <промпт>`)
2. Бот сохраняет ваши медиафайлы в папку ввода ComfyUI
3. Бот загружает API-воркфлоу, подставляет ваш промпт и входные файлы
4. Бот отправляет задачу в ComfyUI через `/api/prompt`
5. Бот опрашивает `/api/history` каждые 10 сек до завершения
6. Результат автоматически отправляется вам в Telegram

---

### Примеры использования

**Анимация фото (самый популярный):**
1. Отправьте фото боту
2. Ответьте на него: `/photo девушка поворачивает голову и улыбается, cinematic`
3. Ждите 2-5 мин

**Видео из текста:**
```
/text кот в космосе на ракете, stars background, epic cinematic
```

**Говорящая голова:**
1. Отправьте портретное фото
2. Ответьте на него: `/talking` + прикрепите голосовое или аудиофайл

**Рестайл видео:**
1. Отправьте видео
2. Ответьте на него: `/v2v anime style, studio ghibli, detailed`

---

### Ограничения

| Ограничение | Описание | Обходной путь |
|:------------|:---------|:-------------|
| Одна задача за раз | T4 не хватает VRAM для параллельной генерации | Используйте `/status` и `/cancel` |
| Таймаут сессии | Colab отключается через 90 мин простоя | Держите вкладку открытой |
| Размер файла | Telegram лимит — 50 МБ | Уменьшите разрешение или длительность |

---

### Решение проблем

| Проблема | Что проверить | Решение |
|:---------|:-------------|:--------|
| Бот не отвечает | Вывод ячейки 5 | Перезапустите ячейку 5, проверьте токен |
| ComfyUI не запускается | `/content/comfyui.log` | Перезапустите ячейки 1-4, затем 5 |
| Генерация зависла | `/status` в боте | `/cancel`, затем попробуйте снова |
| Ошибка нехватки памяти (OOM) | Настройки воркфлоу | Уменьшите разрешение или кадры |
| Туннель не открылся | Вывод ячейки 5 | Перезапустите ячейку 5 |
| Токен не принимается | Формат токена | Должен быть вида `123456:ABCdef...` от @BotFather |
| API-воркфлоу не найден | Вывод ячейки 4 | Перезапустите ячейку 4, проверьте интернет |